# BN â†’ EN Translation Quality Scoring (via Claude API)
Takes an input Excel file with columns `ben`, `ref_en`, `source`,
`translated_en` and uses the **Claude API** to score:

1. **`translated_en` vs `ref_en`** (relative quality) â€” only when `ref_en`
   is present for a row.
2. **`ref_en` vs `ben`** (absolute adequacy of the reference) â€” independent
   judgment of how well the reference translation reflects the Bengali.
3. **`translated_en` vs `ben`** (absolute adequacy of the model output) â€”
   same independent judgment applied to the model's translation.

Each score is 1â€“5 on **adequacy** (meaning preserved) and **fluency**
(natural English), plus a short justification, returned as structured JSON
per row so results are appended as new columns rather than free text.

**Before running:**
1. Kaggle â†’ Add-ons â†’ Secrets â†’ add `ANTHROPIC_API_KEY`.
2. Internet: **On**. No GPU needed â€” this notebook only calls the Claude API.
3. Attach your translated `.xlsx` (output of the bulk-translation notebook)
   via **Add Data â†’ Upload**, then set `INPUT_FILE` below.


## 1. Install dependencies

In [ ]:
!pip install -q pandas openpyxl openai anthropic tenacity


## 2. Secrets + client

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")
#ANTHROPIC_API_KEY = secrets.get_secret("ANTHROPIC_API_KEY")

from openai import OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)
#import anthropic
#client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

OPENAI_MODEL = "gpt-4o-mini"  # cheapest capable model, good for this judgment task
#CLAUDE_MODEL = "claude-sonnet-4-6"

print("OpenAI client ready.")
#print("Anthropic client ready.")


## 3. Load input file

Expects columns: `ben`, `ref_en` (optional), `source` (optional),
`translated_en`. `translated_en` is required â€” this is the output of the
bulk-translation notebook.


In [ ]:
# --- Automation override (added for UI orchestration) ---
import json
import os as _os
import glob as _glob

_RUN_CONFIG_PATH = "/kaggle/input/*/run_config.json"
_matches = sorted(_glob.glob(_RUN_CONFIG_PATH))
_auto_run_label = None
_auto_input_file = None

if _matches:
    _config_path = _matches[-1]
    with open(_config_path) as f:
        _cfg = json.load(f)
    _auto_run_label = _cfg.get("RUN_LABEL")
    _config_dir = _os.path.dirname(_config_path)
    _candidates = (_glob.glob(_config_dir + "/*.xlsx") +
                   _glob.glob(_config_dir + "/*.parquet"))
    if _candidates:
        _auto_input_file = _candidates[0]
    print(f"Automation override found: RUN_LABEL={_auto_run_label}, INPUT_FILE={_auto_input_file}")
else:
    print("No run_config.json found - using manual values below (standalone mode).")


In [ ]:
import os
import glob
import pandas as pd

# EDIT THIS each run: a short label so outputs from different scoring runs never overwrite each other
RUN_LABEL = _auto_run_label or "base_qwen3b"

# EDIT THIS to your actual attached file path
INPUT_FILE = _auto_input_file or "/kaggle/input/your-dataset-name/test_sample_stratified__translated.xlsx"

if not os.path.exists(INPUT_FILE):
    print(f"'{INPUT_FILE}' not found. Files currently under /kaggle/input:")
    for f in glob.glob("/kaggle/input/**/*", recursive=True):
        if os.path.isfile(f):
            print(" ", f)
    print()
    print("Update INPUT_FILE above to match, then re-run this cell.")
else:
    df = pd.read_excel(INPUT_FILE)
    df.columns = [str(c).strip().lower() for c in df.columns]

    if "translated_en" not in df.columns:
        raise ValueError(f"Required column 'translated_en' not found. Columns present: {list(df.columns)}")
    if "ben" not in df.columns:
        raise ValueError(f"Required column 'ben' not found. Columns present: {list(df.columns)}")
    if "ref_en" not in df.columns:
        df["ref_en"] = None
    if "source" not in df.columns:
        df["source"] = "unknown"

    print(f"Loaded {len(df):,} rows.")
    print(f"Rows with ref_en present: {df['ref_en'].notna().sum():,}")
    df.head()


## 4. Scoring prompts

Two separate calls per row, kept independent so the model isn't anchored by
seeing both translations at once when judging absolute adequacy:

- **Absolute scoring** â€” given `ben` + one English candidate, judge
  adequacy (meaning preserved) and fluency (natural English), 1â€“5 each.
  Run once for `ref_en` (if present) and once for `translated_en`.
- **Relative comparison** â€” given `ben`, `ref_en`, and `translated_en`
  together, judge how well `translated_en` matches the meaning of `ref_en`
  specifically (distinct from absolute adequacy vs `ben`), 1â€“5.

All responses are forced to strict JSON so they parse reliably at scale.


In [ ]:
import json

ABSOLUTE_SCORE_SYSTEM = """You are an expert Bengali-English bilingual translation evaluator.
Given a Bengali source sentence and one English candidate translation, score it on:
- adequacy (1-5): how completely and accurately the meaning of the Bengali is preserved
- fluency (1-5): how natural and grammatical the English reads on its own

Respond with ONLY a JSON object, no other text, no markdown fences:
{"adequacy": <int 1-5>, "fluency": <int 1-5>, "notes": "<one short sentence>"}"""

RELATIVE_SCORE_SYSTEM = """You are an expert Bengali-English bilingual translation evaluator.
Given a Bengali source sentence, a reference English translation, and a candidate English
translation produced by a model, score how well the candidate matches the reference in meaning,
independent of the Bengali (i.e. treat the reference as ground truth).
Score match_quality (1-5): 5 = fully equivalent meaning, 1 = unrelated/wrong meaning.

Respond with ONLY a JSON object, no other text, no markdown fences:
{"match_quality": <int 1-5>, "notes": "<one short sentence>"}"""

print("Prompts defined.")


## 5. Robust Claude call with retry + JSON parsing

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

@retry(stop=stop_after_attempt(4), wait=wait_exponential(multiplier=1, min=2, max=20))
def call_llm(system_prompt: str, user_content: str) -> dict:
    resp = client.chat.completions.create(
        model=OPENAI_MODEL,
        max_tokens=300,
        response_format={"type": "json_object"},  # native JSON mode â€” no markdown-fence cleanup needed
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
    )
    text = resp.choices[0].message.content.strip()
    return json.loads(text)

def score_absolute(ben: str, english_candidate: str) -> dict:
    user_content = f"Bengali source:\n{ben}\n\nEnglish candidate:\n{english_candidate}"
    try:
        return call_llm(ABSOLUTE_SCORE_SYSTEM, user_content)
    except Exception as e:
        return {"adequacy": None, "fluency": None, "notes": f"scoring_failed: {e}"}

def score_relative(ben: str, ref_en: str, translated_en: str) -> dict:
    user_content = f"Bengali source:\n{ben}\n\nReference English:\n{ref_en}\n\nCandidate English:\n{translated_en}"
    try:
        return call_llm(RELATIVE_SCORE_SYSTEM, user_content)
    except Exception as e:
        return {"match_quality": None, "notes": f"scoring_failed: {e}"}

print("Scoring functions ready.")


## 6. Run scoring across all rows

Sequential with retry/backoff â€” Claude API calls aren't free, so this
prints running cost-relevant progress (row count) rather than silently
looping. For large files, consider sampling first (Section 6b) before
committing to a full run.


In [ ]:
from tqdm import tqdm

def score_dataframe(frame: pd.DataFrame) -> pd.DataFrame:
    results = []
    for _, row in tqdm(frame.iterrows(), total=len(frame)):
        ben = str(row["ben"])
        ref_en = row["ref_en"]
        translated_en = str(row["translated_en"])

        has_ref = pd.notna(ref_en) and str(ref_en).strip() != ""

        abs_translated = score_absolute(ben, translated_en)

        if has_ref:
            abs_ref = score_absolute(ben, str(ref_en))
            rel = score_relative(ben, str(ref_en), translated_en)
        else:
            abs_ref = {"adequacy": None, "fluency": None, "notes": "no_ref_en"}
            rel = {"match_quality": None, "notes": "no_ref_en"}

        results.append({
            "ref_en_adequacy": abs_ref["adequacy"],
            "ref_en_fluency": abs_ref["fluency"],
            "ref_en_notes": abs_ref.get("notes", ""),
            "translated_en_adequacy": abs_translated["adequacy"],
            "translated_en_fluency": abs_translated["fluency"],
            "translated_en_notes": abs_translated.get("notes", ""),
            "translated_vs_ref_match_quality": rel["match_quality"],
            "translated_vs_ref_notes": rel.get("notes", ""),
        })

    scores_df = pd.DataFrame(results)
    return pd.concat([frame.reset_index(drop=True), scores_df], axis=1)

print("Scoring function ready. Run Section 6b for a quick sample check first, or Section 7 for the full run.")


### 6b. Optional: quick sample check before scoring everything

Recommended first pass â€” scores just 10 rows so you can eyeball output
quality and cost/latency before committing to the full file.


In [ ]:
SAMPLE_CHECK_SIZE = 2
sample_check = score_dataframe(df.head(SAMPLE_CHECK_SIZE))
sample_check[["ben", "ref_en", "translated_en",
              "ref_en_adequacy", "ref_en_fluency",
              "translated_en_adequacy", "translated_en_fluency",
              "translated_vs_ref_match_quality"]]


## 7. Full scoring run

In [ ]:
scored_df = score_dataframe(df)
print("Scoring complete.")
scored_df.head()


## 8. Aggregate summary

In [ ]:
summary = {
    "rows_scored": len(scored_df),
    "rows_with_ref_en": int(scored_df["translated_vs_ref_match_quality"].notna().sum()),
    "avg_translated_en_adequacy": round(scored_df["translated_en_adequacy"].mean(skipna=True), 2),
    "avg_translated_en_fluency": round(scored_df["translated_en_fluency"].mean(skipna=True), 2),
    "avg_ref_en_adequacy": round(scored_df["ref_en_adequacy"].mean(skipna=True), 2) if scored_df["ref_en_adequacy"].notna().any() else None,
    "avg_ref_en_fluency": round(scored_df["ref_en_fluency"].mean(skipna=True), 2) if scored_df["ref_en_fluency"].notna().any() else None,
    "avg_translated_vs_ref_match_quality": round(scored_df["translated_vs_ref_match_quality"].mean(skipna=True), 2) if scored_df["translated_vs_ref_match_quality"].notna().any() else None,
}
print(json.dumps(summary, indent=2))

# Breakdown by source, if multiple sources present
if scored_df["source"].nunique() > 1:
    breakdown = scored_df.groupby("source")[
        ["translated_en_adequacy", "translated_en_fluency", "translated_vs_ref_match_quality"]
    ].mean(numeric_only=True).round(2)
    print()
    print("By source:")
    print(breakdown)


## 9. Save output

In [ ]:
base_name = os.path.splitext(os.path.basename(INPUT_FILE))[0]
OUTPUT_PATH = f"/kaggle/working/{base_name}__scored__{RUN_LABEL}.xlsx"

scored_df.to_excel(OUTPUT_PATH, index=False)

import json as _json
with open(f"/kaggle/working/{base_name}__scoring_summary__{RUN_LABEL}.json", "w") as f:
    _json.dump(summary, f, indent=2)

print(f"Saved scored output to: {OUTPUT_PATH}")
print(f"Saved summary to: /kaggle/working/{base_name}__scoring_summary__{RUN_LABEL}.json")
print()
print("To download to your local machine:")
print("  1. Save this notebook version (Save Version, top right).")
print("  2. Open the notebook's 'Output' tab.")
print("  3. Download the .xlsx and .json files.")


## Notes

- **Cost awareness**: each row makes 1â€“3 Claude API calls. Run Section 6b
  first on a small sample before scoring a full multi-thousand-row file.
- **Retry logic**: `score_absolute`/`score_relative` retry up to 4 times on
  transient failures; persistent failures are recorded as `None` scores
  with a `scoring_failed` note rather than crashing the whole run.
- **Independence of judgments**: `ref_en` and `translated_en` are each
  scored against `ben` in separate calls, so the model isn't anchored by
  seeing one when judging the other â€” only the explicit "relative" call
  compares them directly.
- If you rerun this after fine-tuning, keep the output filename distinct
  (e.g. append `__finetuned`) so before/after scored files don't overwrite
  each other.
